# Predictive DoubleAE codes from audio files

Load a predictive DoubleAE checkpoint, inspect the raw fast latent, slow-derived prediction, residual codec code, and synthesized fast latent, then decode clean, crossed, and corrupted residual/slow codes.

In predictive mode the serialized codes are `(fast_residual, slow)`. The audio decoder receives only the synthesized fast latent.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import math
import sys

repo_root = Path.cwd()
if not (repo_root / "after").exists() and (repo_root.parent / "after").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import cached_conv as cc
import gin
import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
from IPython.display import Audio, display

torch.set_grad_enabled(False)
cc.use_cached_conv(False)

## Paths and options

Update `model_dir` and `audio_paths` for your predictive training run and source files.

In [3]:
# Folder containing the predictive config.gin and checkpoint*.pt files.
model_dir = repo_root / "autoencoder_runs" / "guitar_pred"
checkpoint_step = None  # None selects the latest checkpoint.

audio_paths = {
    "example_a": Path("/data/nils/repos/AFTER/perso/02_Rock1-130-A_comp_mic.wav"),
    "example_b": Path("/data/nils/repos/AFTER/perso/04_Jazz3-150-C_solo_mic.wav"),
}
reference_example = "example_a"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
audio_channels = 1
max_seconds = None  # For example, 10.0 for quicker experiments.
plot_seconds = 1.0
save_outputs = False

## Load and validate the predictive checkpoint

In [4]:
def find_checkpoint(directory, step=None):
    directory = Path(directory).expanduser().resolve()
    if step is not None:
        checkpoint = directory / f"checkpoint{step}.pt"
        if not checkpoint.exists():
            raise FileNotFoundError(checkpoint)
        return checkpoint

    checkpoints = []
    for path in directory.glob("checkpoint*.pt"):
        try:
            checkpoints.append((int(path.stem.replace("checkpoint", "")), path))
        except ValueError:
            pass
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint*.pt files found in {directory}")
    return max(checkpoints, key=lambda item: item[0])[1]


def load_predictive_doubleae(directory, step=None, device="cpu", audio_channels=1):
    directory = Path(directory).expanduser().resolve()
    config_path = directory / "config.gin"
    checkpoint_path = find_checkpoint(directory, step)
    if not config_path.exists():
        raise FileNotFoundError(config_path)

    gin.clear_config()
    gin.enter_interactive_mode()
    gin.parse_config_files_and_bindings([str(config_path)], [])
    with gin.unlock_config():
        gin.bind_parameter("%AUDIO_CHANNELS", audio_channels)

    from after.autoencoder.networks.DoubleNet import DoubleAE

    model = DoubleAE().to(device).eval()
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint.get("model_state", checkpoint)
    incompatible = model.load_state_dict(state_dict, strict=False)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        print("Missing keys:", incompatible.missing_keys)
        print("Unexpected keys:", incompatible.unexpected_keys)

    if not model.predictive_fast_codes:
        raise ValueError("This notebook requires a predictive DoubleAE checkpoint")
    if model.slow_decoder is not None or getattr(model.decoder, "side_channels", 0) != 0:
        raise ValueError("Predictive decoding must not use slow-map conditioning")

    return model, gin.query_parameter("%SR"), checkpoint_path


model, sample_rate, checkpoint_path = load_predictive_doubleae(
    model_dir, checkpoint_step, device=device, audio_channels=audio_channels
)
print(f"Loaded {checkpoint_path}")
print(f"sample_rate={sample_rate}, device={device}")
print(f"fast channels={model.fast_encoder.bottleneck_size}")
print(f"slow channels={model.slow_encoder.bottleneck_size}")
print(f"predictor ratio={model.predictor.upsample_ratio}")

## Load audio examples

In [5]:
def load_audio_tensor(path, sample_rate, audio_channels=1, device="cpu", max_seconds=None):
    path = Path(path).expanduser().resolve()
    if audio_channels == 1:
        wav, _ = librosa.load(path, sr=sample_rate, mono=True, duration=max_seconds)
        wav = wav[None]
    else:
        wav, _ = librosa.load(path, sr=sample_rate, mono=False, duration=max_seconds)
        if wav.ndim == 1:
            wav = np.repeat(wav[None], audio_channels, axis=0)
        elif wav.shape[0] < audio_channels:
            wav = np.repeat(wav[:1], audio_channels, axis=0)
        else:
            wav = wav[:audio_channels]
    return torch.from_numpy(np.asarray(wav, dtype=np.float32)).unsqueeze(0).to(device), path


def audio_to_code_hop(encoder):
    hop = encoder.time_transform.hop_size
    for layer in encoder.down_layers:
        hop *= layer.proj_pool.stride[1]
    return int(hop)


fast_code_hop = audio_to_code_hop(model.fast_encoder)
slow_code_hop = audio_to_code_hop(model.slow_encoder)
examples = {}
target_samples = None
for name, path in audio_paths.items():
    audio, resolved_path = load_audio_tensor(
        path, sample_rate, audio_channels, device, max_seconds
    )
    examples[name] = {"audio": audio, "path": resolved_path}
    target_samples = audio.shape[-1] if target_samples is None else min(target_samples, audio.shape[-1])

# Equal, whole slow-frame lengths make cross-decoding and comparisons unambiguous.
target_samples -= target_samples % slow_code_hop
if target_samples <= 0:
    raise ValueError("Audio must contain at least one slow-code frame")
for example in examples.values():
    example["audio"] = example["audio"][..., :target_samples]

print(f"fast code hop={fast_code_hop} samples ({fast_code_hop / sample_rate:.6f} s)")
print(f"slow code hop={slow_code_hop} samples ({slow_code_hop / sample_rate:.6f} s)")
for name, example in examples.items():
    print(f"{name}: {example['path']} {tuple(example['audio'].shape)}")
    display(Audio(example["audio"][0].detach().cpu().numpy(), rate=sample_rate))

## Encode predictive details

This helper follows the model's predictive encoding path without repeating either encoder. It retains intermediate tensors needed for visualization.

In [6]:
def encode_predictive_details(model, audio):
    fast_raw, fast_kl, fast_multiband, fast_mean = (
        model.fast_encoder._encode_with_multi(audio, return_mean=True)
    )
    slow, slow_kl, slow_multiband = model.slow_encoder._encode_with_multi(audio)
    slow_past = model._shift_slow_to_past(slow)
    prediction = model._predict_from_past(slow_past, fast_raw.shape[-1])
    fast_residual = model.residual_extractor(fast_raw, prediction)
    fast_synth = model.synthesizer(fast_residual, prediction)
    regularisations = {
        "fast_kl": fast_kl,
        "slow_kl": slow_kl,
        "prediction": F.mse_loss(prediction, fast_mean.detach()),
    }
    return {
        "fast_raw": fast_raw,
        "fast_mean": fast_mean,
        "prediction": prediction,
        "fast_residual": fast_residual,
        "fast_synth": fast_synth,
        "slow": slow,
        "slow_past": slow_past,
        "regularisations": regularisations,
        "fast_multiband": fast_multiband,
        "slow_multiband": slow_multiband,
    }


codes = {}
with torch.no_grad():
    for name, example in examples.items():
        codes[name] = encode_predictive_details(model, example["audio"])

for name, code in codes.items():
    print(name)
    for key in ("fast_raw", "prediction", "fast_residual", "fast_synth", "slow", "slow_past"):
        print(f"  {key:14s}: {tuple(code[key].shape)}")
    print("  regularisations:", {
        key: float(value.detach().cpu()) for key, value in code["regularisations"].items()
    })

## Clean reconstruction

Only `fast_residual` and raw `slow` are passed to `decode_codes`.

In [7]:
def to_audio_display(wav):
    return wav.detach().cpu().clamp(-1, 1)[0].numpy()


def display_audio_map(audio_map):
    for name, wav in audio_map.items():
        print(name, tuple(wav.shape))
        display(Audio(to_audio_display(wav), rate=sample_rate))


reconstructions = {}
with torch.no_grad():
    for name, code in codes.items():
        reconstructions[f"{name}_clean"] = model.decode_codes(
            code["fast_residual"], code["slow"]
        )

display_audio_map(reconstructions)

## Decode corrupted residual and slow codes

Noise is scaled independently using each code's standard deviation. Frame dropout removes complete time frames across all channels. `zero_residual` therefore tests prediction-only decoding.

In [ ]:
noise_amount = 0.5
frame_dropout_probability = 0.25
corrupt_span_start_seconds = 0.35
corrupt_span_duration_seconds = 0.15
random_seed = 1234


def add_relative_noise(code, amount, generator):
    scale = code.std(dim=-1, keepdim=True).clamp_min(1e-8)
    noise = torch.randn(code.shape, device=code.device, dtype=code.dtype, generator=generator)
    return code + amount * scale * noise


def drop_frames(code, probability, generator):
    mask = torch.rand(
        code.shape[0], 1, code.shape[-1], device=code.device, generator=generator
    ) < probability
    return torch.where(mask, torch.zeros_like(code), code)


def zero_time_span(code, start_seconds, duration_seconds, code_hop):
    output = code.clone()
    start = int(start_seconds * sample_rate / code_hop)
    length = max(1, math.ceil(duration_seconds * sample_rate / code_hop))
    output[..., start:min(start + length, output.shape[-1])] = 0
    return output


corrupted_decodes = {}
with torch.no_grad():
    for example_index, (name, code) in enumerate(codes.items()):
        generator = torch.Generator(device=code["fast_residual"].device)
        generator.manual_seed(random_seed + example_index)
        residual = code["fast_residual"]
        slow = code["slow"]

        variants = {
            "clean": (residual, slow),
            "zero_residual_prediction_only": (torch.zeros_like(residual), slow),
            # "noisy_residual": (add_relative_noise(residual, noise_amount, generator), slow),
            # "dropped_residual_frames": (drop_frames(residual, frame_dropout_probability, generator), slow),
            # "zero_residual_span": (zero_time_span(residual, corrupt_span_start_seconds, corrupt_span_duration_seconds, fast_code_hop), slow),
            "zero_slow": (residual, torch.zeros_like(slow)),
            "random_slow": (residual, torch.randn(slow.shape, device=slow.device, dtype=slow.dtype, generator=generator)),
            # "noisy_slow": (residual, add_relative_noise(slow, noise_amount, generator)),
            # "dropped_slow_frames": (residual, drop_frames(slow, frame_dropout_probability, generator)),
            # "zero_slow_span": (residual, zero_time_span(slow, corrupt_span_start_seconds, corrupt_span_duration_seconds, slow_code_hop)),
            # "both_noisy": (
            #     add_relative_noise(residual, noise_amount, generator),
            #     add_relative_noise(slow, noise_amount, generator),
            # ),
            # "both_zero": (torch.zeros_like(residual), torch.zeros_like(slow)),
        }
        for variant_name, (variant_residual, variant_slow) in variants.items():
            corrupted_decodes[f"{name}_{variant_name}"] = model.decode_codes(
                variant_residual, variant_slow
            )

display_audio_map(corrupted_decodes)

## Cross residual and slow codes

This tests a residual from one recording with the predictor driven by another recording's slow code.

In [ ]:
cross_decodes = {}
example_names = list(codes)
if len(example_names) >= 2:
    name_a, name_b = example_names[:2]
    with torch.no_grad():
        cross_decodes[f"residual_{name_a}_slow_{name_b}"] = model.decode_codes(
            codes[name_a]["fast_residual"], codes[name_b]["slow"]
        )
        cross_decodes[f"residual_{name_b}_slow_{name_a}"] = model.decode_codes(
            codes[name_b]["fast_residual"], codes[name_a]["slow"]
        )
    display_audio_map(cross_decodes)
else:
    print("Add a second audio path to run cross decoding.")

## Time-aligned waveform and latent plot

The first two raw/full fast channels are overlaid with their predictions. The next two panels show residual channels, followed by the first two raw slow-code channels. Every x-axis is expressed in audio seconds.

In [ ]:
plot_name = reference_example
plot_audio = examples[plot_name]["audio"].detach().cpu()[0]
plot_code = {key: value.detach().cpu() for key, value in codes[plot_name].items() if torch.is_tensor(value)}
seconds = min(plot_seconds, plot_audio.shape[-1] / sample_rate)

audio_time = np.arange(plot_audio.shape[-1]) / sample_rate
fast_time = np.arange(plot_code["fast_raw"].shape[-1]) * fast_code_hop / sample_rate
slow_time = np.arange(plot_code["slow"].shape[-1]) * slow_code_hop / sample_rate
audio_mask = audio_time < seconds
fast_mask = fast_time < seconds
slow_mask = slow_time < seconds

fig, axes = plt.subplots(7, 1, figsize=(16, 14), sharex=True, constrained_layout=True)
axes[0].plot(audio_time[audio_mask], plot_audio[:, audio_mask].mean(0), linewidth=0.8)
axes[0].set_ylabel("audio")
axes[0].set_title(f"{plot_name}: first {seconds:.3f} seconds")

for channel in range(2):
    axis = axes[1 + channel]
    axis.plot(fast_time[fast_mask], plot_code["fast_raw"][0, channel, fast_mask], label=f"full fast {channel}")
    axis.plot(fast_time[fast_mask], plot_code["prediction"][0, channel, fast_mask], "--", alpha=0.85, label=f"prediction {channel}")
    axis.set_ylabel(f"fast {channel}")
    axis.legend(loc="upper right")

for channel in range(2):
    axis = axes[3 + channel]
    axis.plot(fast_time[fast_mask], plot_code["fast_residual"][0, channel, fast_mask], color="tab:green")
    axis.set_ylabel(f"residual {channel}")

for channel in range(2):
    axis = axes[5 + channel]
    axis.step(slow_time[slow_mask], plot_code["slow"][0, channel, slow_mask], where="post", color="tab:purple")
    axis.set_ylabel(f"slow {channel}")

for axis in axes:
    axis.grid(alpha=0.2)
    axis.set_xlim(0, seconds)
axes[-1].set_xlabel("audio time (seconds)")
plt.show()

## Numerical comparison of corruptions

In [ ]:
def mse(a, b):
    length = min(a.shape[-1], b.shape[-1])
    return F.mse_loss(a[..., :length], b[..., :length]).item()


def snr_db(reference, estimate):
    length = min(reference.shape[-1], estimate.shape[-1])
    reference = reference[..., :length]
    error = reference - estimate[..., :length]
    return 10 * torch.log10(reference.square().mean() / error.square().mean().clamp_min(1e-12)).item()


reference_audio = examples[reference_example]["audio"]
clean_audio = corrupted_decodes[f"{reference_example}_clean"]
print(f"{'variant':45s} {'MSE vs input':>14s} {'MSE vs clean':>14s} {'SNR vs input':>14s}")
for name, wav in corrupted_decodes.items():
    if not name.startswith(reference_example + "_"):
        continue
    print(f"{name:45s} {mse(reference_audio, wav):14.6g} {mse(clean_audio, wav):14.6g} {snr_db(reference_audio, wav):14.3f}")

## Optionally save decoded audio

In [ ]:
if save_outputs:
    output_dir = repo_root / "notebooks" / "doubleae_predictive_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    all_outputs = {**reconstructions, **corrupted_decodes, **cross_decodes}
    for name, wav in all_outputs.items():
        torchaudio.save(
            str(output_dir / f"{name}.wav"),
            wav[0].detach().cpu().clamp(-1, 1),
            sample_rate,
        )
    print(f"Saved {len(all_outputs)} files to {output_dir}")
else:
    print("Set save_outputs=True to write WAV files.")